# F2b · gate GT: Qwen3.8-27B pensando contra os decisores (endpoint OpenAI-compatível)

**A pergunta** (issue #75, ADR 0004): o Qwen3.8-27B em modo thinking acerta pelo menos **+10 pontos** a mais que o melhor decisor de um passe, com o piso do IC 95% (bootstrap pareado) **acima de 0**, nas 400 linhas da amostra pública?

Este notebook não precisa de GPU quando você usa um endpoint pronto. Dois modos:
- `MODE = "endpoint"`: chama um servidor OpenAI-compatível (uma API hospedada, ou o seu vLLM).
- `MODE = "local-vllm"`: sobe o vLLM aqui (`vllm==0.31.0`, FP8) numa GPU de uns 40 GB ou mais, como a G4 do Colab (RTX PRO 6000, 96 GB), uma A100 de 80 GB ou uma H100. **Não** serve para a T4 do Kaggle (use `notebooks/decide/think_gt_kaggle.ipynb`).

**Antes de rodar**
1. Suba o `pack.jsonl.gz` (o arquivo **privado** com o texto das provas) para `PACK`. Nunca o torne público, nem este notebook com as saídas.
2. A chave da API vai como segredo do Colab ou do Kaggle, ou na variável de ambiente `KEY_NAME`. Ela nunca é impressa nem vai na linha de comando.
3. **Custo**: tokens de saída da amostra inteira = 400 x (tokens médios por linha) x preço. Rode `LIMIT = "3"` primeiro, leia os tokens por linha no log e multiplique.
4. Se o endpoint responder HTTP 400 a `top_k`, `min_p` ou `chat_template_kwargs`, ponha `STANDARD_ONLY = True` e ligue o pensamento pelo campo do próprio provedor em `EXTRA_BODY`. Isto tira `top_k`, `min_p` e `repetition_penalty` do pedido: a amostragem deixa de ser a do cartão do modelo e o veredito sai `EXPLORATORY (standard_only)`. Os campos do protocolo (`temperature`, `max_tokens`, `chat_template_kwargs`...) não podem ir em `EXTRA_BODY`: o kit recusa.
5. `REVISION`: preencha com o commit do checkpoint **só se** o endpoint serve exatamente esse (o modo `local-vllm` já usa o fixado). Em branco, o veredito sai `EXPLORATORY (revision_not_declared)`: ninguém sabe o que o servidor roda. Um agregador pode mandar cada linha a um upstream ou a uma quantização diferente; o relatório avisa quando as linhas vieram de mais de um endpoint.
6. A chave só viaja por `https://` (ou para `127.0.0.1`), e o kit não segue redirecionamentos: uma URL que redireciona é recusada com o endereço para onde ela aponta.
7. Tudo que este notebook gera é dado de avaliação: **nunca entra em treino** (ADR 0004, item 6).

In [ ]:
# 1. Parâmetros: a única célula que você edita.
SIMPLICIO_SHA = "1b6aea9d71dc5a936765e47871ece2b3cba36453"   # commit de simpletibr/simplicio-27b com decide/think (fixado)
MODE = "endpoint"                         # "endpoint" ou "local-vllm"
PACK = "/content/pack.jsonl.gz"           # privado; no Kaggle: /kaggle/input/<seu-dataset>/pack.jsonl.gz
WORK = "/content/think-gt"                # results.jsonl e report/ ficam aqui
BASE_URL = "https://SEU-ENDPOINT/v1"      # MODE = "endpoint"; em "local-vllm" é preenchido sozinho
MODEL = "Qwen/Qwen3.8-27B-FP8"            # o id que o endpoint conhece
REVISION = ""                             # o que o endpoint roda, se você sabe (vai para o relatório); "" = desconhecido
KEY_NAME = "THINK_API_KEY"                # nome do segredo (Colab/Kaggle) ou da variável de ambiente com a chave
CONCURRENCY = 8                           # requisições em voo
MAX_TOKENS = 16384                        # orçamento de pensamento por pergunta; linha que bate nele conta como erro e é listada
STANDARD_ONLY = False                     # True se o endpoint recusa top_k, min_p e chat_template_kwargs
EXTRA_BODY = ""                           # JSON mesclado em cada requisição (campo do provedor para ligar o thinking), ex.: '{"enable_thinking": true}'
LIMIT = "3"                               # 1a execução: 3 linhas para medir custo e tempo. Depois: "" (todas as pendentes)

from shlex import quote as q      # os valores acima entram nas linhas `!` assim: {q(PACK)}; um espaço, um & ou uma aspa não parte o comando

REV = f"--revision {q(REVISION)}" if REVISION else ""
EXTRA = " ".join(x for x in (f"--limit {q(LIMIT)}" if LIMIT else "", "--standard-only" if STANDARD_ONLY else "",
                             f"--extra-body {q(EXTRA_BODY)}" if EXTRA_BODY else "") if x)

## 2. Código fixado
O código vem do commit `SIMPLICIO_SHA`, nunca de `main`. Esse commit entra no histórico de `main` porque o PR é mergeado com um *merge commit* (não squash); se isso mudar, troque o SHA pelo de um commit que esteja em `main`.

In [ ]:
import os, subprocess

REPO = "/content/simplicio-27b" if os.path.isdir("/content") else os.path.abspath("simplicio-27b")
os.makedirs(REPO, exist_ok=True)
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "init", "-q", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "https://github.com/simpletibr/simplicio-27b.git", SIMPLICIO_SHA], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == SIMPLICIO_SHA, f"checked out {head}, expected {SIMPLICIO_SHA}"
os.chdir(REPO)
print("code at", head)

## 3. Conferências
O pacote tem que ser o da amostra (ids e hashes do `gt-sample-manifest.json`). A chave é lida do segredo ou do ambiente e não é mostrada.

In [ ]:
import sys

assert MODE in ("endpoint", "local-vllm"), f"MODE deve ser 'endpoint' ou 'local-vllm', veio {MODE!r}"
assert MODE == "local-vllm" or "SEU-ENDPOINT" not in BASE_URL, "edite BASE_URL na célula de parâmetros"
assert os.path.exists(PACK), f"{PACK} não existe: suba o pack.jsonl.gz (privado)"
os.makedirs(WORK, exist_ok=True)

key = os.environ.get(KEY_NAME)
if not key:
    try:
        from google.colab import userdata
        key = userdata.get(KEY_NAME)
    except Exception:
        key = None
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret(KEY_NAME)
    except Exception:
        key = None
if key:
    os.environ["THINK_API_KEY"] = key
    print("chave encontrada (não é mostrada)")
else:
    print("sem chave: só serve para um vLLM sem autenticação; uma API hospedada vai responder 401")
del key

In [ ]:
!python -m decide.think.sample verify --manifest decide/think/gt-sample-manifest.json --pack {q(PACK)}

## 4. (só em `MODE = "local-vllm"`) Subir o vLLM
Instala `vllm==0.31.0` e sobe o servidor em `127.0.0.1` (nunca exposto), com o checkpoint FP8 fixado, thinking ligado e o parser `qwen3`. O download tem uns 29 GiB; a primeira subida demora (UNVERIFIED: não medido). O log fica em `WORK/vllm.log`.

In [ ]:
server = None
if MODE == "local-vllm":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.31.0"], check=True)
    from decide.think import common, serve
    revision = REVISION or common.PINNED_REVISIONS.get(MODEL)
    cmd = serve.vllm_command(MODEL, revision)
    log = open(f"{WORK}/vllm.log", "w")
    server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    BASE_URL = serve.base_url()
    REV = f"--revision {q(revision)}" if revision else ""
    assert serve.wait_ready(BASE_URL, 1800, api_key=os.environ.get("THINK_API_KEY")), f"o vLLM não respondeu em 30 min: leia {WORK}/vllm.log"
    print("vLLM no ar em", BASE_URL)

## 5. Rodar o pensador
O pré-voo faz uma chamada de 1 token e confere se o raciocínio volta separado da resposta. Retoma sozinho: se a sessão cair, rode esta célula de novo com os mesmos parâmetros. O orçamento de tokens só pode subir entre execuções.

O log não imprime texto de prova: só o id da linha, tokens, segundos, tokens/s e ETA.

In [ ]:
!python -m decide.think.thinker --backend openai --base-url {q(BASE_URL)} --model {q(MODEL)} {REV} --concurrency {CONCURRENCY} --max-tokens {MAX_TOKENS} --pack {q(PACK)} --manifest decide/think/gt-sample-manifest.json --out {q(WORK)}/results.jsonl {EXTRA}

## 6. Comparar com os 8 decisores
A última linha é o veredito. Com `LIMIT = "3"` ela diz `GT VERDICT: PARTIAL` e o delta que traz é provisório: é só para ver que o relatório monta.

In [ ]:
!python -m decide.think.compare --pack {q(PACK)} --thinker {q(WORK)}/results.jsonl --out {q(WORK)}/report

In [ ]:
if server is not None:
    server.terminate()
    server.wait(timeout=120)
    print("vLLM parado")

## O que fazer com o resultado
- `GT VERDICT: PASS` ou `FAIL` (as duas palavras que valem): só uma execução **completa** do protocolo registrado ganha uma delas: as 400 linhas, o checkpoint fixado, thinking ligado e visto nas linhas, bootstrap e alvo registrados. Linha que não pôde ser respondida depois de várias tentativas vira falha e **conta como erro do modelo**; ela não some da comparação.
- `PARTIAL`: faltam linhas ou há linhas com erro transitório. O delta e o IC da linha são provisórios e **não** valem: rode de novo com os mesmos parâmetros. Com `LIMIT = "3"` é isto que você vê.
- `EXPLORATORY`: fora do protocolo registrado; o motivo vem entre parênteses (`bootstrap_b`, `seed`, `min_delta`, `models_subset`, `pack_not_manifest`, `thinking_off`, `model_not_pinned`, `revision_not_declared`, `standard_only`...). O número é indício, não veredito.
- `INCONCLUSIVE`: o thinking não foi detectado na maioria das linhas (`thinking_not_detected`) ou nada foi pontuado. O número não diz nada sobre o pensador.
- **PASS** com `thinking` ligado e o modelo e a revisão fixados (a primeira linha do relatório diz): o pensador entra no plano (ADR 0004).
- **FAIL com `can still pass`**: o IC ainda comporta +10; mais linhas ou um orçamento maior podem mudar o resultado.
- `AVISO` no relatório (modelo ou revisão diferente do par fixado, thinking desligado, orçamento esgotado, execução parcial) vem antes do número.
- A latência daqui não é a do board (RTX PRO 6000, uma requisição por vez): a seção "Folga de latência" do relatório é um limite superior (UNVERIFIED).
- Não publique `results.jsonl` nem `pack.jsonl.gz`: têm texto de prova e raciocínio sobre ele.